## Single-column RCE model

This notebook contains the structure for your RCE model. There are key functions to complete and all visulizations are left to your design.

This RCE model will follow the Manabe and Strickler 1964 approach.

In [1]:
import numpy as np

#### Main settings and parameters

In [2]:
# Experiment selection:
# 'convective_mode_variation' runs the sweep of equilibrium modes
# 'tau0_variation' runs a sweep of surface optical thickness values (tau0)
select_experiment = 'convective_mode_variation'

# Experiment-dependent settings:
convective_mode_sweep = ["radiative", "dry", "moist"] # the three equilibrium modes to sweep over
tau0_sweep = [0.1, 1, 2, 4, 6] # surface optical depth values to sweep over

#### Parameters

In [35]:
# Physical constants
g = 9.81                  # m/s^2
Rd = 287.0                # J/(kg K)
Rv = 461.0                # J/(kg K)
cp = 1004.0               # J/(kg K)
Lv = 2.5e6                # J/kg
sigma = 5.670374419e-8    # Stefan-Boltzmann
CtoK = 273.15             # Celsius to Kelvin

# Model parameters
Nz = 60                   # number of atmospheric layers
p_surf = 100000.0         # Pa
p_top  = 5000.0           # Pa
dt = 1000.0               # time step (s)
max_iter = 100000          # maximum number of iterations
T0_init = 250.0           # initial isothermal temperature profile (K)

# Radiative transfer settings
S0 = 1361.0                     # solar constant (W/m2)
albedo = 0.3                    # planetary albedo
S_in = S0 * (1 - albedo) / 4.0  # incoming solar radiation at TOA (W/m2)
relh0 = 0.8                     # assumed constant relative humidity for moist convective mixing
tau0 = 4.0                      # optical thickness at surface (tau(p_surf) = tau0)
tau_n = 2.0                     # exponent for tau(p) = tau0 * (p/p0)^n
mu_bar = 0.6                    # diffusivity factor for conversion from I to F
Diff=1.66

#### Helper functions

##### Create vertical grid

In [36]:
'''
This function sets up the vertical grid and initial T-profile for the RCE model.

Its structure is as follows:

             TOP
        -------------------  p_edges[Nz] = p_top
             p_mid[Nz-1]     layer Nz-1
        -------------------  p_edges[Nz-1]
               ...
        -------------------  p_edges[2]
             p_mid[1]        layer 1
        -------------------  p_edges[1]
             p_mid[0]        layer 0
        -------------------  p_edges[0] = p_surf
            SURFACE

Index increases upward; pressure decreases upward.
Grid spacing is uniform in pressure.

Temperature is defined at layer midpoints.
Optical depth tau is defined at edges and decreases upward
(assuming tau0 > 0 and tau_n > 0).

For each layer k:
  p_mid[k] = 0.5 * (p_edges[k] + p_edges[k+1])
  dp[k]    = p_edges[k] - p_edges[k+1] > 0
  dtau[k]  = tau_edges[k] - tau_edges[k+1] > 0

'''  
# above grid from chatGPT == schematic 

def get_grid_tau(Nz, p_surf, p_top, tau0, tau_n, T0_init):

    # Vertical grid (pressure edges and midpoints)
    p_edges = np.linspace(p_surf, p_top, Nz+1)    # edges (Pa), bottom --> top
    p_mid = 0.5*(p_edges[:-1] + p_edges[1:])      # pressure at layer mid-points
    dp = p_edges[:-1] - p_edges[1:]               # dp of each layer (positive)

    # Optical depth at layer edges
    tau_edges = tau0 * (p_edges / p_surf)**tau_n
    dtau = tau_edges[:-1] - tau_edges[1:]

    # Initial isothermal temperature profile (defined at layer midpoints)
    T_init = np.repeat(T0_init, Nz) # K

    grid = {} #make dictionary to hold grid variables
    grid['p_edges'] = p_edges
    grid['p_mid'] = p_mid
    grid['dp'] = dp
    grid['tau_edges'] = tau_edges
    grid['dtau'] = dtau
    grid['T_init'] = T_init

    return grid

##### Thermodynamic functions

In [37]:
# saturation vapor pressure (Magnus)
def es_sat(Tk):
    Tc = Tk - 273.15
    return 610.94 * np.exp(17.625 * Tc / (Tc + 243.04))

# saturation mixing ratio from saturation vapor pressure
def qsat(Tk, p):
    eps = Rd / Rv
    e_s = es_sat(Tk)
    return eps * e_s / (p - (1-eps)*e_s)

def dry_lapse(temp, pres):
    return (Rd/cp)*(temp/pres)

def moist_lapse_rate_K_per_m(DALR, Tk, p, RH=1.0):
    """
    DALR could be subbed in at g/cp for dry adiabatic lapse rate in K/m, or g/(cp*p) for K/Pa.

    Moist-adiabatic lapse rate with saturation mixing ratio scaled by a given RH.
    DALR input is the dry adiabatic lapse rate, provided either in K/m or K/Pa,
    which determines the units of the returned lapse rate.
    """
    q = qsat(Tk, p) * RH
    numerator = DALR * (1.0 + (Lv * q) / (Rd * Tk))
    denominator = 1.0 + (Lv**2 * q) / (cp * Rv * Tk**2)
    return numerator / denominator

##### Radiative tranfer

In [ ]:
# LW FLUXES ONLY 

def longwave_fluxes_and_heating(grid, sfc_temp, temperature):    
   # upwelling and downwelling fluzes 
   # 0 down to sfc, then go back up to TOA 

   F_down = np.zeros(Nz+1)  # downward flux 
   F_up=np.zeros(Nz+1)  # upward flux 
   
   for k in reversed(range(Nz+1)):
      if k==Nz:
            F_down[k] = 0.0  # no downward flux at TOA
      else:
            Tr=np.exp((-1/mu_bar)*(grid['dtau'][k]))
            emis=1-Tr
            F_down[k] = (F_down[k+1] *Tr) + (emis*(sigma*(temperature[k]**4)))

   for k in range(Nz+1):
      if k==0:
            F_up[0] = (sigma*sfc_temp**4) #+S_in # upward flux @ sfc 
      else:
            Tr=np.exp((-1/mu_bar)*(grid['dtau'][k-1]))
            emis=1-Tr
            F_up[k] = F_up[k-1] *Tr +emis*(sigma*temperature[k-1]**4)

   return F_down, F_up
"""

    Your function should conduct downward and upward flux sweeps and return the
    downward (F_down) and upward (F_up) fluxes at the layer edges, which you can
    then use to compute the net flux and heating rate for each layer.

"""

    

'\n\n    Your function should conduct downward and upward flux sweeps and return the\n    downward (F_down) and upward (F_up) fluxes at the layer edges, which you can\n    then use to compute the net flux and heating rate for each layer.\n\n'

In [51]:
grid_ex=get_grid_tau(Nz, p_surf, p_top, tau0, tau_n, T0_init)
F_down,F_up = longwave_fluxes_and_heating(grid_ex, sfc_temp=(S_in/sigma)**.25, temperature=grid_ex['T_init'])
print(F_up)
print(F_down)

[238.175      235.02387734 232.5049226  230.48509993 228.86052353
 227.54984233 226.48917343 225.62821137 224.92723244 224.35478218
 223.88588542 223.50065736 223.18322293 222.92087393 222.70340979
 222.5226209  222.3718823  222.24583369 222.14012661 222.05122413
 221.97624197 221.912822   221.85903124 221.81328127 221.77426341
 221.74089682 221.71228657 221.68768986 221.66648862 221.64816734
 221.632295   221.61851039 221.60651007 221.59603856 221.58688024
 221.57885274 221.57180146 221.56559506 221.56012174 221.55528608
 221.55100655 221.54721329 221.54384636 221.54085422 221.5381925
 221.53582294 221.53371247 221.53183253 221.53015837 221.52866859
 221.52734462 221.5261704  221.52513203 221.52421749 221.52341643
 221.52271997 221.52212053 221.52161169 221.52118809 221.52084532
 221.52057985]
[221.21237614 221.14559629 221.06471166 220.96709568 220.8497123
 220.70906903 220.54116782 220.34145464 220.10476835 219.82529014
 219.49649441 219.11110282 218.66104284 218.13741285 217.530455

##### Convective adjustment

In [80]:
def convective_adjustment(grid,convective_mode, temperature, relh0):
    """
    Adjust temperature profile for convective instability. The approach is to compare
    the T-change between adjacent layers, and if that lapse rate exceeds the dry or
    reference lapse rate, "mix" the two layers by adjusting both layers toward the
    reference lapse rate.

    "Sweeps" refers to the number of times to loop through the vertical profile and
    check for instability. If any mixing occurs, the sweeps continue until either the
    instability is removed or "max_sweeps" is reached. I have max_sweeps set to 10,
    but you can try other values.
    
    """
    max_sweeps = 10
    T_mixed = np.copy(temperature)
    pres=grid['p_mid']
    dp=grid['dp']
    for sweep in range(max_sweeps):  # profiles reach neutral after 10 sweeps or often less
        changed = False
        for k in range(Nz-1):
            dalr=dry_lapse(T_mixed[k], pres[k])
            temp_dalr=dalr*dp[k]
            if convective_mode == 'dry':
                if temp_dalr < (T_mixed[k] - T_mixed[k+1]):
                    T_combined=T_mixed[k] +T_mixed[k+1]
                    T_mixed[k]=(temp_dalr+(T_combined))/2
                    T_mixed[k+1]=(-temp_dalr+(T_combined))/2
                    changed = True
                else: continue

            elif convective_mode == 'moist':
                mlr=moist_lapse_rate_K_per_m(dalr, T_mixed[k], pres[k], RH=relh0)
                temp_mlr=mlr*dp[k]
                if temp_mlr < (T_mixed[k] - T_mixed[k+1]):
                    T_combined=T_mixed[k] +T_mixed[k+1]
                    T_mixed[k]=(temp_mlr+(T_combined))/2
                    T_mixed[k+1]=(-temp_mlr +(T_combined))/2
                    changed = True
                else: continue  

        # If no mixing occurred, end convective sweeps
        if not changed:
            break
    return T_mixed

##### Save model state at a given time step

In [81]:
def save_singlelev_step(history, T_surf, T0, OLR, TOA_net):
    history["T_surf"].append(T_surf)
    history["T0"].append(T0)
    history["OLR"].append(OLR)
    history["TOA_net"].append(TOA_net)
    return history

def save_profiles_step(history, T, F_up_edges, F_down_edges, dTdt_rad, dTdt_conv):
    history["T"].append(T.copy())
    history["F_up"].append(F_up_edges)
    history["F_down"].append(F_down_edges)
    history["dTdt_rad"].append(dTdt_rad)
    history["dTdt_conv"].append(dTdt_conv)
    return history

#### Main RCE driver function

In [82]:
def run_rce_model(grid, convective_mode):

    #### Initialize model state ############################

    # T-profile (K) at layer midpoints
    T = grid['T_init'].copy() # using .copy() avoids modifying the original array

    # Initial surface temp from initial profile
    T_surf = T[0].copy()

    #### Set dictionary arrays for saving model state #################

    history = {
        "T_surf": [], "T0": [], "OLR": [], "TOA_net": [], # Single level stuff
        "T": [], "F_up": [], "F_down": [], "dTdt_rad": [], "dTdt_conv": [] # Nz stuff
        }
        # T_surf = surface temperature (K)
        # T0 = temperature of lowest atmospheric layer (K)
        # OLR = outgoing longwave radiation at TOA (W/m2)
        # TOA_net = net radiation at TOA (W/m2)
        # T = temperature profile (K) at layer midpoints
        # F_up = upward longwave flux (W/m2) at layer edges
        # F_down = downward longwave flux (W/m2) at layer edges
        # dTdt_rad = radiative heating rate (K/s) at layer midpoints
        # dTdt_conv = convective heating rate (K/s) at layer midpoints

    #### Run time loop #####################################
    current_time = 0 # in seconds
    for it in range(max_iter+1):
        # Save prior step temperature profile to get delta and check for convergence
        T_prior = T.copy()

        # Radiative solver
        F_down, F_up = longwave_fluxes_and_heating(grid, temperature=T, sfc_temp=T_surf)

        # Radiative heating rate (K/s) from net flux
        F_net=F_up-F_down # net flux at layer midpoints
        dTdt_rad=(g/cp)*((-np.diff(F_net))/grid['dp'])

        # Apply heating rate to temperature profile
        T+=(dTdt_rad*dt)

        # Update surface temperature
        T_surf = ((S_in + F_down[0])/sigma)**0.25

        # Convective adjustment
        

        # Save current temperature profile to calculate convective heating after convective mixing completes
        dTdt_conv = np.zeros_like(T)
        T_preconv = T.copy()

        if convective_mode != "radiative":

            T = convective_adjustment(grid,convective_mode, T_preconv, relh0)

            # After convective mixing completes, calculate convective heating tendency
            # as difference from pre-convective state
            dTdt_conv = (T - T_preconv)/dt

        current_time += dt

        ########## BOOKKEEPING FROM HERE ##########

        # What you need to do here:
        #   (1) Calculate OLR and TOA_net
        #   (2) Establish a way to check for equilibrium
        #   (3) Complete the write-outs, which are already drafted

        # Maximum temperature change from previous time step
        # This may or may not be useful to you
        maxdT = np.nanmax(np.abs(T - T_prior))

        # Kill if NaN detected
        if np.isnan(maxdT):
            print(f"NaN detected — stopping: it {it:5d} maxdT={maxdT:.5e} K")
            break

        # Save single-level stuff every time step
        F_up_edges=F_up
        F_down_edges=F_down

        OLR = F_up[-1]
        TOA_net = S_in-OLR
        save_singlelev_step(history, T_surf, T[0], OLR, TOA_net)

        # # Write out some stats every nreadstat time step
        nread_stat = 2500
        if it % nread_stat == 0:
            print(f"it {it:5d} maxdT={maxdT:.5e} K, T_surf={T_surf-273:.3f} C, T0={T[0]-273:.3f} C, OLR={OLR:.2f} W/m2, TOA_net={TOA_net:.2f} W/m2")
            #print(F_up[-1])
            #print(F_down[-1])
        # # Save profiles every nprof time step
        nprof = 2500
        if it % nprof == 0:
            save_profiles_step(history, T, F_up_edges, F_down_edges, dTdt_rad, dTdt_conv)

        # Check for equilibrium; break loop if reached
        if TOA_net < 1e-6 and maxdT < 1e-9: 
        #and np.max(np.abs(dTdt_rad)) < 1e-6 and np.max(np.abs(dTdt_conv)) < 1e-6:
            print(f"Equilibrium reached at it {it:5d} maxdT={maxdT:.5e} K, T_surf={T_surf-273:.3f} C, T0={T[0]-273:.3f} C, OLR={OLR:.2f} W/m2, TOA_net={TOA_net:.2f} W/m2")
            
            
            break
    return history

In [78]:
history_save = {} # dictionary to hold all experiment results
grid=get_grid_tau(Nz, p_surf, p_top, tau0, tau_n, T0_init)
history_save['radiative'] = run_rce_model(grid, "radiative")

it     0 maxdT=4.87036e-02 K, T_surf=27.014 C, T0=-23.000 C, OLR=221.50 W/m2, TOA_net=16.68 W/m2
it  2500 maxdT=8.72702e-03 K, T_surf=83.630 C, T0=69.830 C, OLR=185.88 W/m2, TOA_net=52.29 W/m2
it  5000 maxdT=2.89487e-03 K, T_surf=90.928 C, T0=78.189 C, OLR=216.97 W/m2, TOA_net=21.21 W/m2
it  7500 maxdT=1.25753e-03 K, T_surf=92.993 C, T0=80.526 C, OLR=230.46 W/m2, TOA_net=7.71 W/m2
it 10000 maxdT=5.69418e-04 K, T_surf=93.654 C, T0=81.272 C, OLR=235.39 W/m2, TOA_net=2.78 W/m2
it 12500 maxdT=2.70797e-04 K, T_surf=93.877 C, T0=81.523 C, OLR=237.16 W/m2, TOA_net=1.02 W/m2
it 15000 maxdT=1.27318e-04 K, T_surf=93.954 C, T0=81.610 C, OLR=237.80 W/m2, TOA_net=0.38 W/m2
it 17500 maxdT=6.58107e-05 K, T_surf=93.981 C, T0=81.640 C, OLR=238.03 W/m2, TOA_net=0.14 W/m2
it 20000 maxdT=3.25710e-05 K, T_surf=93.991 C, T0=81.652 C, OLR=238.12 W/m2, TOA_net=0.05 W/m2
it 22500 maxdT=1.56239e-05 K, T_surf=93.995 C, T0=81.656 C, OLR=238.15 W/m2, TOA_net=0.02 W/m2
it 25000 maxdT=7.34232e-06 K, T_surf=93.996 C,

### Run model

In [83]:
############## Set up the experiments ##############

# Select the variable to loop over 'loop_var'
# Set a list of experiment names 'exp_names'

# This experiment loops over all equilibrium cases
if select_experiment == 'convective_mode_variation':
    loop_var = convective_mode_sweep
    exp_names = loop_var # Names are just the equilibrium mode names

# This experiment loops over tau cases
elif select_experiment == 'tau0_variation':
    loop_var = tau0_sweep
    exp_names = [f'tau0={tau0 :.1f}' for tau0 in loop_var] # Name them based on actual tau values


############## Loop over the experiments ##############

# Save history from each run and add to dictionary where keys = experiment names
history_save = {}

# Here, iexp is the index of the experiment, isetting is the value of the variable
# being looped over (either convective mode or tau0)
for iexp, isetting in enumerate(loop_var):

        exp_name = exp_names[iexp]

        if select_experiment == 'convective_mode_variation':
            tau0 = 4.0
            convective_mode = isetting
        elif select_experiment == 'tau0_variation':
            tau0 = isetting
            convective_mode = "moist"

        # run GRID to compute new optical depth
        grid = get_grid_tau(Nz, p_surf, p_top, tau0, tau_n, T0_init)

        # --- run model ---
        print()
        print("Starting RCE run for '{}".format(exp_name)+"'")

        history_save[exp_name] = run_rce_model(grid, convective_mode)


Starting RCE run for 'radiative'
it     0 maxdT=4.87036e-02 K, T_surf=27.014 C, T0=-23.000 C, OLR=221.50 W/m2, TOA_net=16.68 W/m2
it  2500 maxdT=8.72702e-03 K, T_surf=83.630 C, T0=69.830 C, OLR=185.88 W/m2, TOA_net=52.29 W/m2
it  5000 maxdT=2.89487e-03 K, T_surf=90.928 C, T0=78.189 C, OLR=216.97 W/m2, TOA_net=21.21 W/m2
it  7500 maxdT=1.25753e-03 K, T_surf=92.993 C, T0=80.526 C, OLR=230.46 W/m2, TOA_net=7.71 W/m2
it 10000 maxdT=5.69418e-04 K, T_surf=93.654 C, T0=81.272 C, OLR=235.39 W/m2, TOA_net=2.78 W/m2
it 12500 maxdT=2.70797e-04 K, T_surf=93.877 C, T0=81.523 C, OLR=237.16 W/m2, TOA_net=1.02 W/m2
it 15000 maxdT=1.27318e-04 K, T_surf=93.954 C, T0=81.610 C, OLR=237.80 W/m2, TOA_net=0.38 W/m2
it 17500 maxdT=6.58107e-05 K, T_surf=93.981 C, T0=81.640 C, OLR=238.03 W/m2, TOA_net=0.14 W/m2
it 20000 maxdT=3.25710e-05 K, T_surf=93.991 C, T0=81.652 C, OLR=238.12 W/m2, TOA_net=0.05 W/m2
it 22500 maxdT=1.56239e-05 K, T_surf=93.995 C, T0=81.656 C, OLR=238.15 W/m2, TOA_net=0.02 W/m2
it 25000 max

## Plots

In [ ]:
#proof of equilibrium


In [ ]:
#plot to match paper

In [ ]:
#SkewT

Deliverables: As usual, you will complete your notebook and push it to your repo by the deadline. In
addition to getting the model completed and running properly, complete each of the following, which entails
a mix of creating visuals and discussing them (in markdown cells):
1. A figure or set of figures that establish that your model is reaching and/or has reached equilibrium,
e.g., from time series and/or snapshot views of certain variables.
2. A skewT with the final equilibrium T -profiles (Teq ) for all three scenarios overlaid.
3. A discussion of how and why the three Teq -profiles differ among the three equilibrium scenarios.
4. A discussion of how the approach to equilibrium differs among the three equilibrium scenarios, and the
physical explanation for these differences.
5. A discussion of some of the choices you had to make in implementing new components in the model
and your process and logic for making those choices.

DISCUSSION